Task 4:

Download the Invoice OCR Dataset from Kaggle and select at least 5 invoice images to work on.
Use Python OCR library Tesseract to extract text from the invoice images and save the extracted text into .txt files.
From the extracted text, identify important information such as Invoice Number, 
Company Name, Date, Customer Name, and Total Amount, then store the results in a pandas dataframe.

https://www.kaggle.com/datasets/senju14/invoice-ocr

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("senju14/invoice-ocr")

print("Path to dataset files:", path)


c:\Users\AhadS\OneDrive\Desktop\github\Nafath\messy notes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1


In [13]:
import os

root = r"C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1"

for folder, _, files in os.walk(root):
    if files:
        print("\nFolder:", folder)
        print("Sample files:", files[:3])


Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\test\annotations
Sample files: ['X00016469619.json', 'X51005230617.json', 'X51005301659.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\test\images
Sample files: ['X00016469619.jpg', 'X51005230617.jpg', 'X51005301659.jpg']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\annotations
Sample files: ['X00016469612.json', 'X00016469620.json', 'X00016469622.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\images
Sample files: ['X00016469612.jpg', 'X00016469620.jpg', 'X00016469622.jpg']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\val\annotations
Sample files: ['X00016469623.json', 'X51005255805.json', 'X51005268200.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\val\images
Sample files: ['X00016469623.jpg', 'X510052

In [ ]:
import os
import re
import pytesseract
import pandas as pd
from PIL import Image

# =========================
# 1. SET TESSERACT PATH
# =========================
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# =========================
# 2. DATASET PATH
# =========================
image_folder = r"C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\images"

# =========================
# 3. STORAGE
# =========================
records = []

# =========================
# 4. LOOP THROUGH IMAGES
# =========================
for file in os.listdir(image_folder):

    if file.lower().endswith((".jpg", ".jpeg", ".png")):

        image_path = os.path.join(image_folder, file)

        # OCR TEXT EXTRACTION
        text = pytesseract.image_to_string(Image.open(image_path))

        # SAVE RAW TEXT FILE
        txt_file = os.path.join(image_folder, file.split(".")[0] + ".txt")
        with open(txt_file, "w", encoding="utf-8") as f:
            f.write(text)

        # =========================
        # 5. EXTRACT INFORMATION
        # =========================

        invoice_no = ""
        date = ""
        total = ""

        inv_match = re.search(r"invoice\s*(?:no|number)?[:#]?\s*([A-Z0-9-]+)", text, re.I)
        if inv_match:
            invoice_no = inv_match.group(1)

        date_match = re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", text)
        if date_match:
            date = date_match.group()

        total_match = re.search(r"(?:total|grand total)[^\d]*([\d,]+\.\d{2})", text, re.I)
        if total_match:
            total = total_match.group(1)

        # =========================
        # 6. BASIC INFO
        # =========================
        lines = [line.strip() for line in text.split("\n") if line.strip()]

        company = lines[0] if len(lines) > 0 else ""
        customer = lines[1] if len(lines) > 1 else ""

        # =========================
        # 7. STORE RECORD
        # =========================
        records.append({
            "File": file,
            "Invoice_Number": invoice_no,
            "Company_Name": company,
            "Date": date,
            "Customer_Name": customer,
            "Total_Amount": total
        })

# =========================
# 8. CREATE DATAFRAME
# =========================
df = pd.DataFrame(records)

# =========================
# 9. DISPLAY OUTPUT
# =========================
from IPython.display import display
display(df)

# =========================
# 10. SAVE RESULT
# =========================
df.to_csv("invoice_results.csv", index=False)

print("DONE ✅ - OCR Completed and CSV saved!")

filling nulls using interpulation 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[["A", "B", "C"]])
data['oridinal_col'] = 
oe.fit_transform(data[['oridinal_col']])

In [2]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# sample data
data = pd.DataFrame({
    "ordinal_col": ["A", "B", "C", "A", "B"]
})

oe = OrdinalEncoder(categories=[["A", "B", "C"]])

data['ordinal_col'] = oe.fit_transform(data[['ordinal_col']])

print(data)

   ordinal_col
0          0.0
1          1.0
2          2.0
3          0.0
4          1.0


In [4]:
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# 1. Load a real dataset (Titanic)
# This dataset has 'age' (numeric) and 'sex'/'embarked' (categorical)
data = sns.load_dataset('titanic')

# We'll use 'age' as the numeric column and 'who' as the categorical column to match the concept
# Let's subset the data for clarity
df = data[['age', 'who']].copy()
df.columns = ['age', 'category']

print("Original Data (First 5 rows):")
print(df.head())
print("\nMissing values before preprocessing:")
print(df.isnull().sum())

# 2. Define the Numeric Transformer
# Steps: Impute missing values with mean, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 3. Define the Categorical Transformer
# Steps: Impute missing values with most frequent, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Combine them into a ColumnTransformer
# Apply numeric_transformer to 'age' and categorical_transformer to 'category'
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, ['age']),
    ('cat', categorical_transformer, ['category'])
])

# 5. Fit and transform the data
preprocessed_data = preprocessor.fit_transform(df)

# 6. Display the result
# Convert back to a DataFrame for easier viewing
# Note: OneHotEncoder creates multiple columns
cat_columns = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(['category'])
all_columns = ['age'] + list(cat_columns)

df_preprocessed = pd.DataFrame(preprocessed_data, columns=all_columns)

print("\nPreprocessed Data (First 5 rows):")
print(df_preprocessed.head())

Original Data (First 5 rows):
    age category
0  22.0      man
1  38.0    woman
2  26.0    woman
3  35.0    woman
4  35.0      man

Missing values before preprocessing:
age         177
category      0
dtype: int64

Preprocessed Data (First 5 rows):
        age  category_child  category_man  category_woman
0 -0.592481             0.0           1.0             0.0
1  0.638789             0.0           0.0             1.0
2 -0.284663             0.0           0.0             1.0
3  0.407926             0.0           0.0             1.0
4  0.407926             0.0           1.0             0.0


In [5]:
import pandas as pd
import mysql.connector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Step 1: Configure database connection
# Step 1 & 2 combined: Connect directly
# Step 1: Configure database connection parameters
db_config = {
    "host": "localhost",
    "user": "root",
    "password": "Q)(RZ5t9",
    "database": "movie_db2"
}

# Step 2: Connect to the database
connection = mysql.connector.connect(**db_config)
cursor = connection.cursor()

# Step 3: Fetch all table names
cursor.execute("SHOW TABLES")
tables = [table[0] for table in cursor.fetchall()]

# Step 4: Define preprocessing pipelines

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Step 5: Process each table
for table in tables:
    print(f"Processing table: {table}")

    # Load table into pandas DataFrame
    query = f"SELECT * FROM `{table}`"
    df = pd.read_sql(query, con=connection)

    # Identify numeric and categorical columns
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Skip if no usable data
    if not numeric_columns and not categorical_columns:
        print(f"Table {table} has no numeric or categorical data to process.")
        continue

    # Define ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_columns),
            ('cat', categorical_transformer, categorical_columns)
        ],
        remainder='passthrough'
    )

    # Fit and transform data
    transformed_data = preprocessor.fit_transform(df)

    # Build feature names
    transformed_columns = []

    # numeric columns
    transformed_columns.extend(numeric_columns)

    # categorical encoded columns
    if categorical_columns:
        ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
        transformed_columns.extend(
            ohe.get_feature_names_out(categorical_columns)
        )

    # passthrough columns
    passthrough_cols = df.columns.difference(numeric_columns + categorical_columns).tolist()
    transformed_columns.extend(passthrough_cols)

    # Convert to DataFrame
    transformed_df = pd.DataFrame(transformed_data, columns=transformed_columns)

    # Create new table
    create_table_query = f"""
    CREATE TABLE `{table}_transformed` (
        {', '.join([f"`{col}` FLOAT" for col in transformed_columns])}
    )
    """

    cursor.execute(f"DROP TABLE IF EXISTS `{table}_transformed`")
    cursor.execute(create_table_query)

    # Insert data
    insert_query = f"""
    INSERT INTO `{table}_transformed`
    VALUES ({', '.join(['%s'] * len(transformed_columns))})
    """

    for row in transformed_df.itertuples(index=False, name=None):
        cursor.execute(insert_query, row)

    print(f"Table {table} transformed and saved as {table}_transformed.")

# Step 6: Commit and close
connection.commit()
cursor.close()
connection.close()

Processing table: a
Table a transformed and saved as a_transformed.
Processing table: a_transformed
Table a_transformed transformed and saved as a_transformed_transformed.
Processing table: a_transformed_transformed


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44

Table a_transformed_transformed transformed and saved as a_transformed_transformed_transformed.
Processing table: a_transformed_transformed_transformed
Table a_transformed_transformed_transformed transformed and saved as a_transformed_transformed_transformed_transformed.
Processing table: a_transformed_transformed_transformed_transformed
Table a_transformed_transformed_transformed_transformed transformed and saved as a_transformed_transformed_transformed_transformed_transformed.
Processing table: aa
Table aa transformed and saved as aa_transformed.
Processing table: aa_transformed


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:48:

Table aa_transformed transformed and saved as aa_transformed_transformed.
Processing table: aa_transformed_transformed
Table aa_transformed_transformed transformed and saved as aa_transformed_transformed_transformed.
Processing table: aa_transformed_transformed_transformed
Table aa_transformed_transformed_transformed transformed and saved as aa_transformed_transformed_transformed_transformed.
Processing table: aa_transformed_transformed_transformed_transformed


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)


Table aa_transformed_transformed_transformed_transformed transformed and saved as aa_transformed_transformed_transformed_transformed_transformed.
Processing table: actor


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\4264465644.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()


ValueError: Shape of passed values is (10, 1), indices imply (10, 22)

In [8]:
import pandas as pd
import mysql.connector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Step 1: Configure database connection parameters as a dictionary
db_config = {
    "host": "localhost",
    "user": "root",
    "password": "Q)(RZ5t9",
    "database": "movie_db2"
}

# Step 2: Connect to the database safely
connection = mysql.connector.connect(**db_config)
cursor = connection.cursor()

# Step 3: Fetch all table names
cursor.execute("SHOW TABLES")
tables = [table[0] for table in cursor.fetchall()]

# Step 4: Define preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Step 5: Process each table
for table in tables:
    # Skip previously processed tables to avoid cyclical running
    if table.endswith('_transformed'):
        continue

    print(f"Processing table: {table}")

    # Load table into pandas DataFrame
    query = f"SELECT * FROM `{table}`"
    df = pd.read_sql_query(query, con=connection)

    # Identify numeric and categorical columns
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Skip if no usable data
    if not numeric_columns and not categorical_columns:
        print(f"Table '{table}' has no numeric or categorical data to process. Skipping...")
        continue

    # Define ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_columns),
            ('cat', categorical_transformer, categorical_columns)
        ],
        remainder='passthrough'
    )

    # Fit and transform data
    transformed_data = preprocessor.fit_transform(df)
    
    # Handle sparse matrix variations securely
    if hasattr(transformed_data, "toarray"):
        transformed_data = transformed_data.toarray()

    # Dynamically extract original feature names from the transformer pipeline
    raw_columns = preprocessor.get_feature_names_out()
    
    # Clean up scikit-learn prefixes ('num__' and 'cat__')
    transformed_columns = [col.split('__')[-1] for col in raw_columns]

    # Convert spaces, brackets, and periods to underscores for clean SQL strings
    base_cleaned_columns = [
        col.replace(' ', '_').replace('.', '_').replace('[', '').replace(']', '') 
        for col in transformed_columns
    ]

    # --- ADVANCED DEDUPLICATION LOGIC: Fixes case-insensitive and passthrough duplicates ---
    cleaned_columns = []
    seen_lower_names = {}

    for col in base_cleaned_columns:
        col_lower = col.lower()  # Force lowercase to match MySQL's engine behavior
        
        if col_lower in seen_lower_names:
            seen_lower_names[col_lower] += 1
            # Append suffix to make it completely unique to MySQL
            unique_col = f"{col}_{seen_lower_names[col_lower]}"
        else:
            seen_lower_names[col_lower] = 0
            unique_col = col
            
        cleaned_columns.append(unique_col)
    # --------------------------------------------------------------------------------------

    # Convert transformed numpy array safely to a DataFrame
    transformed_df = pd.DataFrame(transformed_data, columns=cleaned_columns)

    # Build SQL creation columns schema
    create_columns_schema = ', '.join([f"`{col}` FLOAT" for col in cleaned_columns])
    create_table_query = f"""
    CREATE TABLE `{table}_transformed` (
        {create_columns_schema}
    )
    """

    # Drop existing copy and recreate the structural table schema
    cursor.execute(f"DROP TABLE IF EXISTS `{table}_transformed`")
    cursor.execute(create_table_query)

    # Build bulk dynamic insert parameter query
    insert_query = f"""
    INSERT INTO `{table}_transformed`
    VALUES ({', '.join(['%s'] * len(cleaned_columns))})
    """

    # Iterate rows and stream insert sanitized entries to MySQL
    for row in transformed_df.itertuples(index=False, name=None):
        sanitized_row = [None if pd.isna(val) else val for val in row]
        cursor.execute(insert_query, sanitized_row)

    print(f"Table '{table}' transformed successfully and saved as '{table}_transformed'.")

# Step 6: Commit changes and properly close background infrastructure
connection.commit()
cursor.close()
connection.close()
print("\nPipeline execution finished completely without any errors!")

Processing table: a
Table 'a' transformed successfully and saved as 'a_transformed'.
Processing table: aa
Table 'aa' transformed successfully and saved as 'aa_transformed'.
Processing table: actor


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816

Table 'actor' transformed successfully and saved as 'actor_transformed'.
Processing table: director
Table 'director' transformed successfully and saved as 'director_transformed'.
Processing table: gener
Table 'gener' transformed successfully and saved as 'gener_transformed'.
Processing table: movie_actor


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816

Table 'movie_actor' transformed successfully and saved as 'movie_actor_transformed'.
Processing table: movie_director
Table 'movie_director' transformed successfully and saved as 'movie_director_transformed'.
Processing table: movie_gener


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816

Table 'movie_gener' transformed successfully and saved as 'movie_gener_transformed'.
Processing table: moviec
Table 'moviec' transformed successfully and saved as 'moviec_transformed'.
Processing table: n
Table 'n' transformed successfully and saved as 'n_transformed'.
Processing table: prod


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816

Table 'prod' transformed successfully and saved as 'prod_transformed'.
Processing table: quots
Table 'quots' transformed successfully and saved as 'quots_transformed'.
Processing table: s
Table 's' transformed successfully and saved as 's_transformed'.

Pipeline execution finished completely without any errors!


C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, con=connection)
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\AhadS\AppData\Local\Temp\ipykernel_15700\3115881816

Steps:
1. Extract data from each store’s POS system (CSV files) Store's data

2. ransform using Pipline:
Missing values:Fill or drop rows for missing Qty, Unit_Price, CustomerID
Data type conversion:	Qty → int, Unit_Price → float, SaleDate → datetime
Calculate new columns: Total_Price = Qty * Unit_Price
Currency conversion: Convert prices to OMR
Case formatting: Trim and normalize text fields

3. Load the cleaned data into a structured SQL Server database 

In [1]:
"""
ETL Pipeline - Store Sales (Week 11 Task)
==========================================
1. EXTRACT  - Load all 3 store CSV files
2. TRANSFORM - Clean data, fix types, compute Total_Price, convert to OMR
3. LOAD     - Save cleaned data to a single CSV (simulated "SQL load")
"""

import pandas as pd
import numpy as np

# ============================================================
# STEP 1: EXTRACT
# ============================================================
files = [
    "store_sales_1.csv",
    "store_sales_2.csv",
    "store_sales_3.csv",
]

dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f"[EXTRACT] Total rows loaded: {len(raw)}")
print(f"[EXTRACT] Columns: {list(raw.columns)}\n")

# ============================================================
# STEP 2: TRANSFORM
# ============================================================

# --- 2a. Text normalisation ---
raw["StoreID"]       = raw["StoreID"].str.strip().str.upper().str.replace("-", "_")
raw["CurrencyType"]  = raw["CurrencyType"].str.strip().str.upper()
raw["ProductName"]   = raw["ProductName"].str.strip().str.title()

# Filter out obvious junk rows in ProductName (single words that aren't product names)
JUNK_NAMES = {"Line", "Actually", "Case", "Black", "On", "Soon", "Skill"}
before = len(raw)
raw = raw[~raw["ProductName"].isin(JUNK_NAMES)]
print(f"[TRANSFORM] Removed {before - len(raw)} junk rows by ProductName.")

# --- 2b. Data type conversion ---
raw["Qty"]        = pd.to_numeric(raw["Qty"],        errors="coerce")
raw["Unit_Price"] = pd.to_numeric(raw["Unit_Price"], errors="coerce")
raw["SaleDate"]   = pd.to_datetime(raw["SaleDate"],  errors="coerce", dayfirst=False)

# Qty should be integer – we'll handle after imputation
# CustomerID keep as string

# --- 2c. Handle missing values ---
# Strategy:
#   Qty        → fill with median (reasonable for units sold)
#   Unit_Price → fill with median per ProductName, else global median
#   CustomerID → fill with "UNKNOWN"
#   CurrencyType → fill with "USD" (most common / default assumption)
#   SaleDate   → drop rows where date is missing (unusable records)

# Drop rows with no SaleDate
before = len(raw)
raw = raw.dropna(subset=["SaleDate"])
print(f"[TRANSFORM] Dropped {before - len(raw)} rows with missing SaleDate.")

# Impute Unit_Price: per-product median first, then global
global_price_median = raw["Unit_Price"].median()
raw["Unit_Price"] = raw.groupby("ProductName")["Unit_Price"].transform(
    lambda x: x.fillna(x.median())
)
raw["Unit_Price"] = raw["Unit_Price"].fillna(global_price_median)

# Impute Qty: per-product median, then global
global_qty_median = raw["Qty"].median()
raw["Qty"] = raw.groupby("ProductName")["Qty"].transform(
    lambda x: x.fillna(x.median())
)
raw["Qty"] = raw["Qty"].fillna(global_qty_median)

# Convert Qty to int after imputation
raw["Qty"] = raw["Qty"].round().astype(int)

# Fill missing CustomerID and CurrencyType
raw["CustomerID"]   = raw["CustomerID"].fillna("UNKNOWN")
raw["CurrencyType"] = raw["CurrencyType"].fillna("USD")

# --- 2d. Currency conversion → OMR ---
# Exchange rates (approximate, fixed for this ETL)
USD_TO_OMR = 0.385
CONVERSION = {"USD": USD_TO_OMR, "OMR": 1.0}

def to_omr(row):
    rate = CONVERSION.get(row["CurrencyType"], USD_TO_OMR)  # default USD if unknown
    return round(row["Unit_Price"] * rate, 4)

raw["Unit_Price_OMR"] = raw.apply(to_omr, axis=1)

# --- 2e. Calculate Total_Price (in OMR) ---
raw["Total_Price_OMR"] = (raw["Qty"] * raw["Unit_Price_OMR"]).round(4)

# --- 2f. Final column order & rename ---
clean = raw[[
    "StoreID",
    "ProductName",
    "Qty",
    "Unit_Price",
    "CurrencyType",
    "Unit_Price_OMR",
    "Total_Price_OMR",
    "SaleDate",
    "CustomerID",
]].copy()

print(f"\n[TRANSFORM] Clean dataset shape: {clean.shape}")
print(f"[TRANSFORM] Remaining nulls:\n{clean.isnull().sum()}\n")

# ============================================================
# STEP 3: LOAD
# ============================================================
output_path = "store_sales_cleaned.csv"
clean.to_csv(output_path, index=False)
print(f"[LOAD] Saved cleaned data → {output_path}")

# ============================================================
# SUMMARY REPORT
# ============================================================
print("\n========== SUMMARY ==========")
print(f"Total records loaded : {len(raw)}")
print(f"Date range           : {clean['SaleDate'].min().date()} → {clean['SaleDate'].max().date()}")
print(f"Stores               : {sorted(clean['StoreID'].unique())}")
print(f"Currencies found     : {sorted(raw['CurrencyType'].unique())}")
print(f"Total Revenue (OMR)  : {clean['Total_Price_OMR'].sum():,.2f}")
print(f"Avg Order Value (OMR): {clean['Total_Price_OMR'].mean():.2f}")
print("\nTop 5 Products by Revenue (OMR):")
top = (
    clean.groupby("ProductName")["Total_Price_OMR"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
for prod, rev in top.items():
    print(f"  {prod:<30} {rev:>8.2f} OMR")
print("==============================\n")
print(clean.head(10).to_string(index=False))

[EXTRACT] Total rows loaded: 300
[EXTRACT] Columns: ['ProductName', 'Qty', 'Unit_Price', 'SaleDate', 'CurrencyType', 'CustomerID', 'StoreID']

[TRANSFORM] Removed 7 junk rows by ProductName.
[TRANSFORM] Dropped 0 rows with missing SaleDate.

[TRANSFORM] Clean dataset shape: (293, 9)
[TRANSFORM] Remaining nulls:
StoreID            0
ProductName        0
Qty                0
Unit_Price         0
CurrencyType       0
Unit_Price_OMR     0
Total_Price_OMR    0
SaleDate           0
CustomerID         0
dtype: int64

[LOAD] Saved cleaned data → store_sales_cleaned.csv

========== SUMMARY ==========
Total records loaded : 293
Date range           : 2024-05-25 → 2025-05-25
Stores               : ['STORE_A']
Currencies found     : ['OMR', 'USD']
Total Revenue (OMR)  : 5,477.32
Avg Order Value (OMR): 18.69

Top 5 Products by Revenue (OMR):
  Moore Chair                      141.97 OMR
  Gray Tool                        118.80 OMR
  Flores Coat                      105.40 OMR
  Mitchell Frame     

In [ ]:
# =========================================================
# COMPLETE MULTI-STORE ETL PIPELINE
# =========================================================
# FIXED VERSION
#
# PROBLEM SOLVED:
# Duplicate StoreID values such as:
# STORE_A
# store-A
# Store_A
#
# SOLUTION:
# Generate UNIQUE IDs for:
# - Product
# - Customer
# - Store
# - Sale
#
# Even if CSV files do not contain IDs
# =========================================================

# =========================================================
# INSTALL LIBRARIES
# =========================================================
# pip install pandas sqlalchemy pymysql scikit-learn

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import glob
import os
import uuid

from sqlalchemy import create_engine, text

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# =========================================================
# DATABASE CONNECTION
# =========================================================

username = "root"
password = "Q)(RZ5t9"
host = "localhost"
database = "storeN"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

# =========================================================
# EXTRACT FUNCTION
# =========================================================

def extract_data(folder_path):

    print("\n==============================")
    print("EXTRACTING CSV FILES")
    print("==============================")

    csv_files = glob.glob(
        os.path.join(folder_path, "*.csv")
    )

    all_dataframes = []

    for file in csv_files:

        print(f"Reading File: {file}")

        df = pd.read_csv(file)

        # Add source filename
        df["Source_File"] = os.path.basename(file)

        all_dataframes.append(df)

    combined_data = pd.concat(
        all_dataframes,
        ignore_index=True
    )

    print("\nFILES MERGED SUCCESSFULLY")

    print(combined_data.head())

    return combined_data

# =========================================================
# TRANSFORM FUNCTION
# =========================================================

def transform_data(data):

    print("\n==============================")
    print("TRANSFORMING DATA")
    print("==============================")

    # 1. REMOVE DUPLICATES
    data = data.drop_duplicates()

    # 2. HANDLE MISSING VALUES
    print("\nHandling Missing Values...")
    data["Qty"] = data["Qty"].fillna(data["Qty"].mean())
    data["Unit_Price"] = data["Unit_Price"].fillna(data["Unit_Price"].mean())
    data = data.dropna(subset=["CustomerID"])

    # 3. DATA TYPE CONVERSION
    print("Converting Data Types...")
    data["Qty"] = data["Qty"].astype(int)
    data["Unit_Price"] = data["Unit_Price"].astype(float)
    data["CustomerID"] = data["CustomerID"].astype(str)
    data["StoreID"] = data["StoreID"].astype(str)
    
    # FIXED: Added mixed format parsing
    data["SaleDate"] = pd.to_datetime(data["SaleDate"], format="mixed")

    # 4. TEXT CLEANING
    print("Cleaning Text Fields...")
    text_columns = ["ProductName", "CurrencyType", "CustomerID", "StoreID"]
    for col in text_columns:
        data[col] = data[col].astype(str).str.strip().str.lower()

    # 5. GENERATE UNIQUE PRODUCT IDS
    print("Generating Product IDs...")
    unique_products = data["ProductName"].unique()
    product_mapping = {product: str(uuid.uuid4()) for product in unique_products}
    data["ProductID"] = data["ProductName"].map(product_mapping)

    # 6. GENERATE UNIQUE STORE IDS
    print("Generating Store IDs...")
    unique_stores = data["StoreID"].unique()
    store_mapping = {store: str(uuid.uuid4()) for store in unique_stores}
    data["GeneratedStoreID"] = data["StoreID"].map(store_mapping)

    # 7. GENERATE UNIQUE CUSTOMER IDS
    print("Generating Customer IDs...")
    unique_customers = data["CustomerID"].unique()
    customer_mapping = {customer: str(uuid.uuid4()) for customer in unique_customers}
    data["GeneratedCustomerID"] = data["CustomerID"].map(customer_mapping)

    # 8. GENERATE SALE IDs
    print("Generating Sale IDs...")
    data["SaleID"] = [str(uuid.uuid4()) for _ in range(len(data))]

    # 9. CURRENCY CONVERSION
    print("Converting Currency to OMR...")
    exchange_rates = {"usd": 0.385, "eur": 0.420, "omr": 1}
    data["Unit_Price_OMR"] = data.apply(
        lambda row: row["Unit_Price"] * exchange_rates.get(row["CurrencyType"], 1),
        axis=1
    )

    # 10. CREATE TOTAL PRICE (FIXED COLUMN NAME TO MATCH EXPECTED LOAD OUTPUT)
    print("Creating Total_Price...")
    data["Total_Price"] = data["Qty"] * data["Unit_Price_OMR"]

    # REMOVED: The sklearn pipeline step has been taken out completely 
    # to protect your real metrics from being converted to machine learning Z-scores.

    print("\nTRANSFORMATION COMPLETED")
    print(data[["ProductName", "Qty", "Unit_Price_OMR", "Total_Price", "SaleDate"]].head())

    return datadef transform_data(data):

    print("\n==============================")
    print("TRANSFORMING DATA")
    print("==============================")

    # 1. REMOVE DUPLICATES
    data = data.drop_duplicates()

    # 2. HANDLE MISSING VALUES
    print("\nHandling Missing Values...")
    data["Qty"] = data["Qty"].fillna(data["Qty"].mean())
    data["Unit_Price"] = data["Unit_Price"].fillna(data["Unit_Price"].mean())
    data = data.dropna(subset=["CustomerID"])

    # 3. DATA TYPE CONVERSION
    print("Converting Data Types...")
    data["Qty"] = data["Qty"].astype(int)
    data["Unit_Price"] = data["Unit_Price"].astype(float)
    data["CustomerID"] = data["CustomerID"].astype(str)
    data["StoreID"] = data["StoreID"].astype(str)
    
    # FIXED: Added mixed format parsing
    data["SaleDate"] = pd.to_datetime(data["SaleDate"], format="mixed")

    # 4. TEXT CLEANING
    print("Cleaning Text Fields...")
    text_columns = ["ProductName", "CurrencyType", "CustomerID", "StoreID"]
    for col in text_columns:
        data[col] = data[col].astype(str).str.strip().str.lower()

    # 5. GENERATE UNIQUE PRODUCT IDS
    print("Generating Product IDs...")
    unique_products = data["ProductName"].unique()
    product_mapping = {product: str(uuid.uuid4()) for product in unique_products}
    data["ProductID"] = data["ProductName"].map(product_mapping)

    # 6. GENERATE UNIQUE STORE IDS
    print("Generating Store IDs...")
    unique_stores = data["StoreID"].unique()
    store_mapping = {store: str(uuid.uuid4()) for store in unique_stores}
    data["GeneratedStoreID"] = data["StoreID"].map(store_mapping)

    # 7. GENERATE UNIQUE CUSTOMER IDS
    print("Generating Customer IDs...")
    unique_customers = data["CustomerID"].unique()
    customer_mapping = {customer: str(uuid.uuid4()) for customer in unique_customers}
    data["GeneratedCustomerID"] = data["CustomerID"].map(customer_mapping)

    # 8. GENERATE SALE IDs
    print("Generating Sale IDs...")
    data["SaleID"] = [str(uuid.uuid4()) for _ in range(len(data))]

    # 9. CURRENCY CONVERSION
    print("Converting Currency to OMR...")
    exchange_rates = {"usd": 0.385, "eur": 0.420, "omr": 1}
    data["Unit_Price_OMR"] = data.apply(
        lambda row: row["Unit_Price"] * exchange_rates.get(row["CurrencyType"], 1),
        axis=1
    )

    # 10. CREATE TOTAL PRICE (FIXED COLUMN NAME TO MATCH EXPECTED LOAD OUTPUT)
    print("Creating Total_Price...")
    data["Total_Price"] = data["Qty"] * data["Unit_Price_OMR"]

    # REMOVED: The sklearn pipeline step has been taken out completely 
    # to protect your real metrics from being converted to machine learning Z-scores.

    print("\nTRANSFORMATION COMPLETED")
    print(data[["ProductName", "Qty", "Unit_Price_OMR", "Total_Price", "SaleDate"]].head())

    return data

# =========================================================
# CREATE TABLES FUNCTION
# =========================================================

def create_tables():

    print("\n==============================")
    print("CREATING TABLES")
    print("==============================")

    with engine.connect() as conn:

        # -------------------------------------------------
        # PRODUCT TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Product (

            ProductID VARCHAR(255) PRIMARY KEY,

            ProductName VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # CUSTOMER TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Customer (

            CustomerPK VARCHAR(255) PRIMARY KEY,

            OriginalCustomerID VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # STORE TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Store (

            StorePK VARCHAR(255) PRIMARY KEY,

            OriginalStoreID VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # SALE TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Sale (

            SaleID VARCHAR(255) PRIMARY KEY,

            ProductID VARCHAR(255),

            CustomerPK VARCHAR(255),

            StorePK VARCHAR(255),

            Qty FLOAT,

            Unit_Price_OMR FLOAT,

            Total_Price FLOAT,

            SaleDate DATE,

            CurrencyType VARCHAR(50),

            Source_File VARCHAR(255),

            FOREIGN KEY (ProductID)
            REFERENCES Product(ProductID),

            FOREIGN KEY (CustomerPK)
            REFERENCES Customer(CustomerPK),

            FOREIGN KEY (StorePK)
            REFERENCES Store(StorePK)

        )

        """))

        conn.commit()

    print("TABLES CREATED SUCCESSFULLY")

# =========================================================
# LOAD FUNCTION
# =========================================================

def load_data(data):

    print("\n==============================")
    print("LOADING DATA")
    print("==============================")

    # -----------------------------------------------------
    # PRODUCT TABLE
    # -----------------------------------------------------

    product_df = data[[

        "ProductID",
        "ProductName"

    ]].drop_duplicates()

    # -----------------------------------------------------
    # CUSTOMER TABLE
    # -----------------------------------------------------

    customer_df = data[[

        "GeneratedCustomerID",
        "CustomerID"

    ]].drop_duplicates()

    customer_df.columns = [

        "CustomerPK",
        "OriginalCustomerID"

    ]

    # -----------------------------------------------------
    # STORE TABLE
    # -----------------------------------------------------

    store_df = data[[

        "GeneratedStoreID",
        "StoreID"

    ]].drop_duplicates()

    store_df.columns = [

        "StorePK",
        "OriginalStoreID"

    ]

    # -----------------------------------------------------
    # SALE TABLE
    # -----------------------------------------------------

    sale_df = data[[

        "SaleID",

        "ProductID",

        "GeneratedCustomerID",

        "GeneratedStoreID",

        "Qty",

        "Unit_Price_OMR",

        "Total_Price",

        "SaleDate",

        "CurrencyType",

        "Source_File"

    ]]

    sale_df.columns = [

        "SaleID",

        "ProductID",

        "CustomerPK",

        "StorePK",

        "Qty",

        "Unit_Price_OMR",

        "Total_Price",

        "SaleDate",

        "CurrencyType",

        "Source_File"

    ]

    # -----------------------------------------------------
    # LOAD DATA
    # -----------------------------------------------------

    product_df.to_sql(
        "Product",
        con=engine,
        if_exists="append",
        index=False
    )

    customer_df.to_sql(
        "Customer",
        con=engine,
        if_exists="append",
        index=False
    )

    store_df.to_sql(
        "Store",
        con=engine,
        if_exists="append",
        index=False
    )

    sale_df.to_sql(
        "Sale",
        con=engine,
        if_exists="append",
        index=False
    )

    print("DATA LOADED SUCCESSFULLY")

# =========================================================
# VERIFY FUNCTION
# =========================================================

def verify_data():

    print("\n==============================")
    print("VERIFYING DATA")
    print("==============================")

    query = """

    SELECT *
    FROM Sale

    """

    result = pd.read_sql(query, engine)

    print(result.head())

# =========================================================
# MAIN ETL FUNCTION
# =========================================================

# =========================================================
# MAIN ETL FUNCTION
# =========================================================

def run_etl_pipeline():

    # STEP 1 — CHANGED "store_files" TO "."
    data = extract_data(".")

    # STEP 2
    transformed_data = transform_data(data)

    # STEP 3
    create_tables()

    # STEP 4
    load_data(transformed_data)

    # STEP 5
    verify_data()

    print("\nETL PIPELINE COMPLETED SUCCESSFULLY")

# =========================================================
# RUN PIPELINE
# =========================================================

run_etl_pipeline()








EXTRACTING CSV FILES
Reading File: .\store_sales_1.csv
Reading File: .\store_sales_2.csv
Reading File: .\store_sales_3.csv
Reading File: .\store_sales_cleaned.csv

FILES MERGED SUCCESSFULLY
          ProductName  Qty  Unit_Price    SaleDate CurrencyType  \
0         Smith Paper  3.0        10.5   7/13/2024          OMR   
1      Johnson Screen  NaN         NaN   2/23/2025          Usd   
2  Roberts Ingredient  3.0        30.0  11/13/2024          USD   
3       White Monitor  NaN        10.5   4/16/2025          USD   
4  Rodriguez Keyboard  2.0        20.0    8/3/2024          usd   

                             CustomerID  StoreID        Source_File  \
0  9ca482a2-0356-49c1-b5e3-88ae98d1cc2f  Store_A  store_sales_1.csv   
1  c0b9df4e-8f03-4bf0-a31b-0a7d7c2a8907  Store_A  store_sales_1.csv   
2  97dc18e3-2c12-4e26-9863-32514e82e822  Store_A  store_sales_1.csv   
3  e4d09733-d496-47b3-a4b5-04de84d8fd06  Store_A  store_sales_1.csv   
4  435ecb46-4545-4af7-b72c-119f64d193a5  Store_A  s

OperationalError: (pymysql.err.OperationalError) (1049, "Unknown database 'storen'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)